# Structural Identifiability — Jalisco Measles SEIR + Vaccination Model
### *Annex analysis using `StructuralIdentifiability.jl`*

This notebook assesses the structural identifiability of the transmission model used in the Jalisco measles vaccination counterfactual. Structural identifiability determines whether model parameters can, in principle, be uniquely recovered from perfect (noise-free) observations of the model output — a property of the model equations themselves, independent of data quality.

The fitted model is age-structured (five bands), but **structural identifiability is governed by the equation *structure*, not the number of bands** — each band repeats the same S→E→I→R flow with frequency-dependent transmission and per-band reporting, coupled through the contact matrix. We therefore analyse the **single-band form of the fitted model**; the age-structured model inherits these identifiability properties per band.

We assess two scenarios:
- **A — reported incidence only, unknown initial conditions** (the hardest case)
- **B — known initial conditions and known population** (the conditions of the fitted analysis, where baseline susceptibility is fixed from serosurvey data and population from official projections)

---
## ⚙️ First: switch the runtime to Julia
**Runtime → Change runtime type → Julia**, then run the setup cell below. (Colab ships Julia as a native runtime — no installer needed.)

In [1]:
# One-time setup for this session. First run precompiles the CAS stack (~2–4 min); later runs are instant.
using Pkg
Pkg.add("StructuralIdentifiability")
using StructuralIdentifiability

versioninfo()
Pkg.status("StructuralIdentifiability")

    Updating registry at `~/.julia/registries/General.toml`
   Resolving package versions...
   Installed OpenBLAS32_jll ──────────── v0.3.33+2
   Installed TimerOutputs ────────────── v0.5.29
   Installed FLINT_jll ───────────────── v301.400.1+0
   Installed ParamPunPam ─────────────── v0.5.7
   Installed RationalFunctionFields ──── v0.3.1
   Installed Combinatorics ───────────── v1.1.0
   Installed RandomExtensions ────────── v0.4.4
   Installed Nemo ────────────────────── v0.54.2
   Installed StructuralIdentifiability ─ v0.5.25
   Installed AbstractAlgebra ─────────── v0.48.6
   Installed Groebner ────────────────── v0.10.3
  Installing 2 artifacts
   Installed artifact OpenBLAS32               10.3 MiB
   Installed artifact FLINT                    23.6 MiB
    Updating `~/.julia/environments/v1.12/Project.toml`
  [220ca800] + StructuralIdentifiability v0.5.25
    Updating `~/.julia/environments/v1.12/Manifest.toml`
⌅ [c3fe647b] + AbstractAlgebra v0.48.6
  [861a8166] + Combinatoric

Julia Version 1.12.6


   6848.5 ms  ✓ StructuralIdentifiability
  13 dependencies successfully precompiled in 434 seconds. 482 already precompiled.


Commit 15346901f00 (2026-04-09 19:20 UTC)
Build Info:
  Official https://julialang.org release
Platform Info:
  OS: Linux (x86_64-linux-gnu)
  CPU: 2 × Intel(R) Xeon(R) CPU @ 2.20GHz
  WORD_SIZE: 64
  LLVM: libLLVM-18.1.7 (ORCJIT, broadwell)
  GC: Built with stock GC
Threads: 2 default, 1 interactive, 2 GC (on 2 virtual cores)
Environment:
  JULIA_NUM_THREADS = auto
Status `~/.julia/environments/v1.12/Project.toml`
  [220ca800] StructuralIdentifiability v0.5.25


## How to read the output

`assess_identifiability` tags every parameter and state with one of three verdicts:

| Verdict | Meaning |
|---|---|
| `:globally` | Uniquely recoverable from perfect, noise-free output. |
| `:locally` | Pinned to a finite set of discrete values, not a single one. |
| `:nonidentifiable` | Infinitely many values reproduce the same output. |

**Syntax rules**
- Inside `@ODEmodel`, every state carries its time argument: `S(t)`, not `S`.
- In `known_ic`, list *bare* state names: `known_ic = [S, E, I, R]`.
- `@ODEmodel` rejects decimals — use integer ratios.

**Model structure.** `beta` is the effective transmission rate (`q` × own-band contact rate); `sigma = 1/latent`; `gamma = 1/infectious`; `rho` is the reporting fraction; `N` is the band population (entering via frequency-dependence). Vaccination enters as a **known time-varying input** `u(t)` (administered doses × efficacy), so it introduces no unknown parameters. The observable is **reported incidence** `y1(t) = rho·sigma·E(t)` — what surveillance counts.

> ⏱️ The *first* `assess_identifiability` call may sit for ~25–30 s while it computes the IO-equations and Wronskians. That is expected — do not interrupt the kernel.

> 🧩 **States vs. parameters.** A *state* can be `:nonidentifiable` while every parameter is fine — usually an unobserved or cumulative compartment whose starting value is not pinned down. When we say a model is "structurally identifiable," we mean the **parameters**.

---
## Scenario A — reported incidence only, **unknown** initial conditions

The single output carries the reporting fraction only inside the product `rho·sigma·E`, and the transmission rate enters only as `beta/N`. This is the hardest case and mirrors the classic under-reporting structure.

In [2]:
ode_A = @ODEmodel(
    S'(t) = -beta*S(t)*I(t)/N - u(t)*S(t),
    E'(t) =  beta*S(t)*I(t)/N - sigma*E(t),
    I'(t) =  sigma*E(t) - gamma*I(t),
    R'(t) =  gamma*I(t) + u(t)*S(t),
    y1(t) =  rho*sigma*E(t)
)
assess_identifiability(ode_A)

[ Info: Summary of the model:
[ Info: State variables: S, E, I, R
[ Info: Parameters: N, beta, gamma, rho, sigma
[ Info: Inputs: u
[ Info: Outputs: y1
[ Info: Assessing local identifiability
[ Info: Assessing global identifiability
[ Info: Note: the input model has nontrivial submodels. If the computation for the full model will be too heavy, you may want to try to first analyze one of the submodels. They can be produced using function `find_submodels`
[ Info: Computing IO-equations
[ Info: Computed IO-equations in 9.846011293 seconds
[ Info: Computing Wronskians
[ Info: Computed Wronskians in 2.025969776 seconds
[ Info: Dimensions of the Wronskians [32]
[ Info: Ranks of the Wronskians computed in 0.029986219 seconds
[ Info: Global identifiability assessed in 26.143185763 seconds


OrderedCollections.OrderedDict{Any, Symbol} with 9 entries:
  S(t)  => :nonidentifiable
  E(t)  => :nonidentifiable
  I(t)  => :nonidentifiable
  R(t)  => :nonidentifiable
  N     => :nonidentifiable
  beta  => :nonidentifiable
  gamma => :globally
  rho   => :nonidentifiable
  sigma => :globally

**Reading the result.** The progression rates `sigma` and `gamma` return `:globally` identifiable, but the transmission rate `beta`, the population size `N`, and — the quantity modellers care about most — the reporting fraction `rho` are all `:nonidentifiable`. The four states are `:nonidentifiable` as well, the usual consequence of unknown initial conditions. In plain terms: **the true infection burden cannot be recovered from reported cases alone.** Because `rho` is not identifiable, the reported curve is equally consistent with a small, well-reported outbreak and a large, poorly-reported one. Breaking this tie requires extra information — known initial conditions, an externally fixed reporting fraction, or a second data stream such as serology.

In [3]:
# Which parameter COMBINATIONS survive when the individual parameters do not:
find_identifiable_functions(ode_A)

[ Info: Computing IO-equations
[ Info: Computed IO-equations in 0.026164134 seconds
[ Info: Computing Wronskians
[ Info: Computed Wronskians in 0.038256269 seconds
[ Info: Dimensions of the Wronskians [32]
[ Info: Ranks of the Wronskians computed in 9.7279e-5 seconds
[ Info: Simplifying generating set. Simplification level: standard
✓ # Computing specializations..     Time: 0:00:06
[ Info: Search for polynomial generators concluded in 7.175877291
[ Info: Selecting generators in 0.488128045
[ Info: Inclusion checked with probability 0.995 in 0.084614808 seconds
[ Info: The search for identifiable functions concluded in 41.156609324 seconds


3-element Vector{AbstractAlgebra.Generic.FracFieldElem{Nemo.QQMPolyRingElem}}:
 sigma
 gamma
 (N*rho)//beta

**Reading the result.** Alongside `sigma` and `gamma`, the only surviving transmission-related quantity is the grouped combination `(N*rho)/beta`. The three parameters `beta`, `N`, and `rho` reach the observed output *only* through this single group, so none can be recovered on its own — estimating any one requires fixing the others. This is the algebraic root of the under-reporting problem.

---
## Scenario B — **known** initial conditions and known population

In the fitted analysis the baseline susceptible fraction (hence the initial states) is fixed from the serosurvey, and `N` is fixed from population projections. We encode *N known* by adding `y2 = N`, and pass `known_ic` for all states.

In [4]:
ode_B = @ODEmodel(
    S'(t) = -beta*S(t)*I(t)/N - u(t)*S(t),
    E'(t) =  beta*S(t)*I(t)/N - sigma*E(t),
    I'(t) =  sigma*E(t) - gamma*I(t),
    R'(t) =  gamma*I(t) + u(t)*S(t),
    y1(t) =  rho*sigma*E(t),
    y2(t) =  N
)
assess_identifiability(ode_B, known_ic = [S, E, I, R])

[ Info: Summary of the model:
[ Info: State variables: S, E, I, R
[ Info: Parameters: N, beta, gamma, rho, sigma
[ Info: Inputs: u
[ Info: Outputs: y1, y2
[ Info: Computing IO-equations
[ Info: Computed IO-equations in 1.215363432 seconds
[ Info: Computing Wronskians
[ Info: Computed Wronskians in 0.044639581 seconds
[ Info: Dimensions of the Wronskians [32, 2]
[ Info: Ranks of the Wronskians computed in 0.000120084 seconds
[ Info: Simplifying generating set. Simplification level: standard
[ Info: Search for polynomial generators concluded in 0.198882854
[ Info: Selecting generators in 0.01433704
[ Info: Inclusion checked with probability 0.99875 in 0.025347206 seconds
[ Info: The search for identifiable functions concluded in 2.89342635 seconds
[ Info: The search for identifiable functions with known initial conditions concluded in 2.925685574 seconds
[ Info: Assessing identifiability with known initial conditions concluded in 11.990154018 seconds


OrderedCollections.OrderedDict{Nemo.QQMPolyRingElem, Symbol} with 9 entries:
  S(0)  => :globally
  E(0)  => :globally
  I(0)  => :globally
  R(0)  => :globally
  N     => :globally
  beta  => :globally
  gamma => :globally
  rho   => :globally
  sigma => :globally

**Reading the result.** With the initial conditions and population size treated as known, *every* parameter — including `beta`, `N`, and `rho` — flips to `:globally` identifiable. Fixing the initial states and `N` resolves the `(N*rho)/beta` entanglement and renders the full parameter set identifiable. This is the decisive point for the analysis: **the model as estimated — with baseline susceptibility fixed from serosurvey data and population fixed from projections — is structurally identifiable.** It also explains why the counterfactual is reported in averted *confirmed* cases: the reporting fraction is only identifiable once the initial conditions are fixed, so the reported-case scale is the appropriate and defensible quantity to report.

---
## Summary

| Parameter | Scenario A (unknown ICs) | Scenario B (known ICs, known N) |
|---|---|---|
| `sigma` (1/latent) | globally identifiable | globally identifiable |
| `gamma` (1/infectious) | globally identifiable | globally identifiable |
| `beta` (transmission) | non-identifiable | globally identifiable |
| `N` (population) | non-identifiable | globally identifiable |
| `rho` (reporting fraction) | non-identifiable | globally identifiable |

Under Scenario A the non-identifiable parameters enter the output only through the combination `(N*rho)/beta`. Under Scenario B — the conditions of the fitted analysis — all parameters are globally identifiable. Non-identifiability here arises from unknown initial conditions and a single reported-incidence output, and is resolved by fixing the initial states and population, not by collecting cleaner data.